# IV measurements

Clean notebook for source-measure IV sweeps.

### Workflow
1. **Browse** for the project root (folder containing the `Instruments` package).
2. **Scan** VISA instruments — each resource is listed with its type (Keithley 2450/2470, Yokogawa GS200/7651, ...).
3. **Assign** the Bias instrument (required) and the Gate instrument (optional).
   For Keithley 2450/2470 you are also asked for FRONT or REAR terminals.

### Measurement modes included
1. **Single IV** — one IV sweep for quick tests.
2. **Gate loop IV** — IV sweep at each Gate voltage.

### Important choices
- Bias IV uses **NPLC = 1**.
- Gate loop pattern: `configure → output_on → voltage_ramping → hysteresis_voltage_sweep`.
- Magnetic-field readings are not part of this notebook.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import csv
import time
import sys
import pyvisa
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from tkinter import Tk, filedialog

# ---- Project root: browse for it ----
# Pick the folder that contains the Instruments package.
# Falls back to a typed path if the dialog is cancelled or unavailable.
PROJECT_ROOT = None
try:
    _tk = Tk()
    _tk.withdraw()
    _tk.attributes("-topmost", True)
    _chosen = filedialog.askdirectory(
        title="Select project root (folder containing the Instruments package)",
        initialdir=str(Path.cwd()),
    )
    _tk.destroy()
    if _chosen:
        PROJECT_ROOT = Path(_chosen)
except Exception as exc:
    print("Folder picker unavailable:", exc)

while PROJECT_ROOT is None:
    typed = input("Type the project root path: ").strip().strip('"')
    if typed:
        PROJECT_ROOT = Path(typed)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



from Instruments.keithley2450_with_add_ons import Keithley2450
from Instruments.yokogawags200_with_add_ons import YokogawaGS200
from Instruments.yokogawa7651_with_add_ons import Yokogawa7651

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root does not exist: {PROJECT_ROOT} - re-run this cell and pick again."
    )

In [ ]:
# ============================================================
# 2. Scan connected VISA instruments and identify their type
# ============================================================
rm = pyvisa.ResourceManager()
connected_instruments = rm.list_resources()

# USB vendor IDs, used when an instrument does not answer *IDN?
USB_VENDORS = {
    "0x05E6": "Keithley",
    "0x0B21": "Yokogawa",
}

def classify_instrument(idn, resource_name):
    """Return a short instrument-type string from an *IDN? reply."""
    vendor = "Unknown"
    for vid, name in USB_VENDORS.items():
        if vid in resource_name:
            vendor = name
    if not idn:
        return vendor + " (no *IDN? reply)"
    u = idn.upper()
    if "KEITHLEY" in u or vendor == "Keithley":
        for model in ("2450", "2470", "2604", "2600", "2000", "2182", "6221", "2700"):
            if model in u:
                return f"Keithley {model}"
        return "Keithley (unknown model)"
    if "YOKOGAWA" in u or vendor == "Yokogawa":
        for model in ("GS200", "GS210", "7651"):
            if model in u:
                return f"Yokogawa {model}"
        return "Yokogawa (unknown model)"
    if "STANFORD" in u or "SR8" in u:
        return "SRS lock-in amplifier"
    return idn.split(",")[0] or "Unknown"

instrument_list = []  # list of (resource, idn_or_None, type)

print("Connected VISA resources:")
print("-" * 78)
for res in connected_instruments:
    idn = None
    try:
        inst = rm.open_resource(res, timeout=2000)
        try:
            inst.clear()
        except Exception:
            pass
        idn = inst.query("*IDN?").strip()
        inst.close()
    except Exception:
        pass
    typ = classify_instrument(idn, res)
    instrument_list.append((res, idn, typ))
    print(f"{res}  [{typ}]")
    if idn:
        print(f"    IDN: {idn}")
print("-" * 78)

if not instrument_list:
    print("No VISA instruments found.")

## Instrument assignment

After the VISA scan, the connection cell asks you to assign instruments:

- **Bias** — required. Choosing none raises an error.
- **Gate** — optional. Press Enter to measure without a Gate;
  Gate calls are then ignored and Gate metadata is recorded as NaN.

If a Keithley **2450/2470** is chosen (Bias or Gate), you are also asked
whether to use the **FRONT or REAR terminals**.

**Yokogawa instruments can be used for the Gate only** (via a small adapter:
`voltage_ramping`, `output_on/off`, `measure__voltage`, `configure_voltage_source`).
The Bias must be a Keithley 2450/2470 — choosing a Yokogawa as Bias raises an error.

In [ ]:
# ============================================================
# 3. Assign Bias and Gate instruments
# ============================================================
# Bias is REQUIRED: not choosing one raises an error.
# Gate is OPTIONAL: press Enter to run without a Gate.


class _NoGate:
    """Placeholder used when no Gate instrument is selected.
    Every call is ignored (warned once); measure functions return NaN."""

    _warned = False

    def _warn(self):
        if not _NoGate._warned:
            print("NOTE: no Gate instrument selected - Gate call ignored.")
            _NoGate._warned = True

    def write(self, *a, **k):
        self._warn()

    def output_on(self):
        self._warn()

    def output_off(self):
        self._warn()

    def voltage_ramping(self, *a, **k):
        self._warn()

    def measure__voltage(self):
        return float("nan")

    def measure__current(self):
        return float("nan")

    def configure_voltage_source(self, *a, **k):
        self._warn()


class _YokogawaGateAdapter:
    """Adapt YokogawaGS200 / Yokogawa7651 to the Keithley-style Gate calls
    used in this notebook."""

    def __init__(self, yoko):
        self._y = yoko

    # raw SCPI passthrough (terminal commands do not exist on Yokogawa)
    def write(self, cmd):
        if cmd.strip().upper().lstrip(":").startswith("ROUT"):
            return
        self._y.write(cmd)

    def ask(self, cmd):
        return self._y.ask(cmd)

    def output_on(self):
        if hasattr(self._y, "enable_source"):      # 7651
            self._y.enable_source()
        else:                                       # GS200
            self._y.source_enabled = True

    def output_off(self):
        if hasattr(self._y, "disable_source"):     # 7651
            self._y.disable_source()
        else:                                       # GS200
            self._y.source_enabled = False

    def voltage_ramping(self, target_voltage, step_size=1, pause=20e-3, callback=None):
        # same semantics as Keithley2450.voltage_ramping: step in mV, pause in s
        self._y.ramping_with_monitor(
            float(target_voltage), abs(step_size) * 1e-3, pause
        )

    def measure__voltage(self):
        if hasattr(self._y, "source_level"):       # GS200
            return float(self._y.source_level)
        return float(self._y.source_voltage)       # 7651

    def measure__current(self):
        # ponytail: GS200/7651 current readback needs their measure cycle; not needed for gate
        return float("nan")

    def configure_voltage_source(self, nplc=1, current=1e-6,
                                 auto_range=False, compliance_current=None):
        if hasattr(self._y, "apply_voltage"):      # 7651
            self._y.apply_voltage(
                max_voltage=30,
                compliance_current=compliance_current or 10e-3,
            )
        else:                                       # GS200
            self._y.source_mode = "voltage"
            if compliance_current is not None:
                # driver truncates to the manual range 1 mA - 100 mA
                self._y.current_limit = compliance_current


def _choose_instrument(role, required):
    print(f"\nSelect the {role} instrument from the scanned list:")
    for i, (res, idn, typ) in enumerate(instrument_list, start=1):
        print(f"  {i}: {res}  [{typ}]")
    while True:
        choice = input(f"{role} - enter a number (or press Enter for none): ").strip()
        if choice == "":
            if required:
                print(f"A {role} instrument is required - you must choose one.")
                continue
            return None
        try:
            res, idn, typ = instrument_list[int(choice) - 1]
            return res, typ
        except (ValueError, IndexError):
            print("Invalid choice - try again.")


def _ask_terminals(role):
    t = input(f"{role} is a Keithley 2450/2470 - FRONT or REAR terminals? ").strip().lower()
    while t not in ("front", "fron", "f", "rear", "r"):
        t = input("Please type 'front' or 'rear': ").strip().lower()
    return "FRON" if t in ("front", "fron", "f") else "REAR"


def _open_instrument(resource, typ):
    """Open a resource with the driver matching its detected type."""
    if "2450" in typ or "2470" in typ or "Keithley" in typ:
        return Keithley2450(resource), "keithley"
    if "7651" in typ:
        return _YokogawaGateAdapter(Yokogawa7651(resource)), "yokogawa"
    if "GS200" in typ or "GS210" in typ or "Yokogawa" in typ:
        return _YokogawaGateAdapter(YokogawaGS200(resource)), "yokogawa"
    print(f"WARNING: unrecognized type '{typ}' - opening with the Keithley 2450 driver.")
    return Keithley2450(resource), "keithley"


if not instrument_list:
    raise RuntimeError("No VISA instruments found - cannot assign Bias/Gate.")

# ---- Bias (required) ----
_bias = _choose_instrument("Bias", required=True)
if _bias is None:
    raise RuntimeError("A Bias instrument is required - aborting.")
BIAS_VISA, BIAS_TYPE = _bias

if "Yokogawa" in BIAS_TYPE:
    raise RuntimeError(
        "Yokogawa instruments can only be used for gating - "
        "the Bias must be a Keithley 2450/2470."
    )

Bias, BIAS_DRIVER = _open_instrument(BIAS_VISA, BIAS_TYPE)
print("Bias connected:", BIAS_VISA, f"[{BIAS_TYPE}]")

if "2450" in BIAS_TYPE or "2470" in BIAS_TYPE:
    term = _ask_terminals("Bias")
    if term == "REAR":
        Bias.use_rear_terminals()
    else:
        Bias.use_front_terminals()
    print("Bias terminals set to:", Bias.check_terminals())

# ---- Gate (optional) ----
_gate = _choose_instrument("Gate", required=False)
if _gate is None:
    GATE_VISA, GATE_TYPE, GATE_DRIVER = None, None, None
    Gate = _NoGate()
    print("No Gate instrument - Gate calls will be ignored.")
else:
    GATE_VISA, GATE_TYPE = _gate
    Gate, GATE_DRIVER = _open_instrument(GATE_VISA, GATE_TYPE)
    print("Gate connected:", GATE_VISA, f"[{GATE_TYPE}]")
    if GATE_DRIVER == "keithley" and ("2450" in GATE_TYPE or "2470" in GATE_TYPE):
        term = _ask_terminals("Gate")
        if term == "REAR":
            Gate.use_rear_terminals()
        else:
            Gate.use_front_terminals()
        print("Gate terminals set to:", Gate.check_terminals())


# Manual control — Gate and Bias

These cells are intentionally simple, like the control cells in your original notebook.  
Run only the cell you need.


In [ ]:
# ============================================================
# Configure Gate voltage source
# ============================================================

def configure_gate(nplc=1, current_range_A=5e-9, auto_range=False,
                   compliance_A=5e-9):
    """Configure the Gate as a voltage source.

    Keithley 2450/2470: uses the existing configure_voltage_source,
    and makes sure the terminal selection (FRONT/REAR) is preserved.

    Yokogawa: asks the user for the source limit, per the instrument manual.
    """
    if GATE_VISA is None:
        print("No Gate instrument - nothing to configure.")
        return

    if GATE_DRIVER == "keithley":
        before = Gate.check_terminals()
        Gate.configure_voltage_source(
            nplc=nplc,
            current=current_range_A,
            auto_range=auto_range,
            compliance_current=compliance_A,
        )
        after = Gate.check_terminals()
        if after != before:
            # make sure the terminal used stays the same as before configuring
            if before == "REAR":
                Gate.use_rear_terminals()
            else:
                Gate.use_front_terminals()
            print(f"Terminals restored to {before} after configuring.")
        else:
            print(f"Terminals unchanged ({before}).")
        print("Gate configured.")

    elif GATE_DRIVER == "yokogawa":
        if "7651" in (GATE_TYPE or ""):
            print("Yokogawa 7651 manual: voltage mode ranges 10 mV / 100 mV / 1 V / 10 V / 30 V;")
            print("compliance current 1 mA - 120 mA.")
            max_v = float(input("  Max source voltage [V]: "))
            limit = float(input("  Compliance current limit [A]: "))
            Gate._y.apply_voltage(max_voltage=max_v, compliance_current=limit)
        else:
            print("Yokogawa GS200 manual: in voltage mode the current limit is")
            print("SOURce:PROTection:CURRent, valid range 1 mA - 100 mA.")
            limit = float(input("  Current limit [A]: "))
            Gate._y.source_mode = "voltage"
            Gate._y.current_limit = limit
            print("GS200 current limit now:", Gate._y.current_limit, "A")
        print("Gate configured.")


configure_gate()

## Keithley buffer / communication cleanup

Use these only if a sweep was interrupted or you received a communication/buffer error.


In [ ]:
# Clean up leftover Bias sweep buffer
Bias.adapter.connection.clear()
time.sleep(1)

try:
    Bias.write('TRAC:DEL "sweepBuffer"')
except Exception as exc:
    print("Buffer delete message:", exc)

time.sleep(1)

try:
    print("Bias error:", Bias.ask(":SYST:ERR?"))
except Exception as exc:
    print("Could not query Bias error:", exc)


In [ ]:
# ============================================================
# BIAS RECOVERY — close VISA session and reconnect
# ============================================================

import time
import gc

print("Starting Bias recovery...")

try:
    Bias.adapter.connection.close()
    print("Old Bias VISA session closed.")
except Exception as exc:
    print("Could not close old Bias session:", exc)

try:
    del Bias
except:
    pass

gc.collect()
time.sleep(2)

Bias = Keithley2450(BIAS_VISA)

time.sleep(1)

print("Bias IDN:", Bias.ask("*IDN?"))

Bias.write("*CLS")
time.sleep(0.5)

print("Bias Error:", Bias.ask(":SYST:ERR?"))

Bias.write(":ROUT:TERM REAR")

print("Bias recovery completed.")

In [ ]:
# Clear Bias communication/error queue and identify instrument
Bias.adapter.connection.clear()
time.sleep(1)

Bias.write("*CLS")
time.sleep(0.5)

print("IDN:", Bias.ask("*IDN?"))
print("Error:", Bias.ask(":SYST:ERR?"))


In [ ]:
# Optional Gate communication/error cleanup (Keithley driver only)
if GATE_VISA is not None and GATE_DRIVER == "keithley":
    Gate.adapter.connection.clear()
    time.sleep(1)

    Gate.write("*CLS")
    time.sleep(0.5)

    print("Gate IDN:", Gate.ask("*IDN?"))
    print("Gate Error:", Gate.ask(":SYST:ERR?"))
else:
    print("No Keithley Gate - nothing to clean up.")

# IV sweep function

The **measurement action** below follows the original working ICE IV function:

```python
Bias.create_hysteresis_voltage_sweep_config_list(...)
Bias.write(":SENS:CURR:NPLC 1")
output = Bias.run_clist_sweep(delay=0.0)
```

The first three CSV columns remain `Bias, Current, Time`, and the Gate
metadata column is appended afterward.

### Filename convention
Every measurement filename contains:
- Gate voltage
- Bias range

The folder structure is flat: one measurement folder can contain the entire dataset.

In [ ]:
# ============================================================
# IV sweep — based on the working ICE function
# ============================================================

def hysteresis_voltage_sweep(
        bias1,
        bias2,
        step,
        max_current,
        save_dir
):
    """
    Full hysteresis IV:
        0 -> +bias1 -> bias2 -> 0

    bias1, bias2, step are in mV.
    """
    # Same sweep mechanism as the working ICE notebook
    Bias.create_hysteresis_voltage_sweep_config_list(
        bias1,
        bias2,
        step,
        max_current
    )

    Bias.write(":SENS:CURR:NPLC 1")

    output = Bias.run_clist_sweep(delay=0.0)

    output = output.split(",")
    data = [float(x) for x in output]

    curr = np.asarray(data[0::3], dtype=float)
    bias = np.asarray(data[1::3], dtype=float)
    rel_time = np.asarray(data[2::3], dtype=float)

    # Metadata AFTER the sweep; this does not change how the IV is acquired.
    try:
        gate_voltage = float(Gate.measure__voltage())
    except Exception:
        gate_voltage = np.nan

    gate_col = np.full(len(bias), gate_voltage)

    # Keep original Bias/Current/Time first.
    data_to_save = np.column_stack((
        bias,
        curr,
        rel_time,
        gate_col
    ))

    os.makedirs(save_dir, exist_ok=True)

    timestamp = int(time.time())

    # Put Gate and Bias range directly in the filename.
    # This lets ALL measurements live in one folder and makes MATLAB parsing easy.
    gate_tag = f"{gate_voltage:+.3f}" if np.isfinite(gate_voltage) else "NA"

    filename_core = (
        f"VI{timestamp}"
        f"_Gate{gate_tag}V"
        f"_BiasFrom{bias1}To{bias2}mV"
    )

    data_filename = os.path.join(
        save_dir,
        filename_core + ".csv"
    )

    figure_filename = os.path.join(
        save_dir,
        filename_core + ".png"
    )

    np.savetxt(
        data_filename,
        data_to_save,
        delimiter=",",
        header=(
            "Bias (V),"
            "Current (A),"
            "Time (s),"
            "Gate (V)"
        ),
        comments=""
    )

    plt.figure()
    plt.plot(bias, curr)
    plt.xlabel("Bias (V)")
    plt.ylabel("Current (A)")

    plt.title(f"Hysteresis Voltage | Gate = {gate_voltage:.4g} V")
    plt.grid()
    plt.savefig(
        figure_filename,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    print("Saved:", data_filename)

    return bias, curr, rel_time

# MODE 1 — Single IV test

Use this for quick checks.  
Set the Gate manually with the control cells above if you want a particular Gate voltage.


In [ ]:
# Re-run the Gate configuration (terminal-safe, handles Yokogawa)
configure_gate(nplc=1, current_range_A=5e-9, compliance_A=5e-9)

In [ ]:
# ============================================================
# MODE 1: SINGLE IV
# ============================================================

single_bias_positive_mV = 450
single_bias_negative_mV = -450
single_bias_step_mV = 5
single_max_current_A = 100e-6
time.sleep(2)
Gate.output_on()
time.sleep(2)
hysteresis_voltage_sweep(
    single_bias_positive_mV,
    single_bias_negative_mV,
    single_bias_step_mV,
    single_max_current_A,
    SINGLE_IV_SAVE_DIR
)
time.sleep(2)
Gate.output_off()

In [ ]:
Gate.voltage_ramping(
    0.0,
    10,
    0.1
)
Gate.output_off()


# MODE 2 — IV loop at different Gate voltages

This is deliberately written in the same style as your working loop.

In [ ]:
np.linspace(-2, 2.5, 91)

In [ ]:
np.linspace(-2, 4, 241)

In [ ]:
Gate.measure__voltage()

In [ ]:
# ============================================================
# MODE 2: GATE LOOP IV
# ============================================================

GATE_LOOP_SAVE_DIR = (
    r"C:\Users\ICE\Desktop\ICE Measurements\Renu\R037\MoS2 DB 2L\ICE\tunnel\Middle to Top\Top gate\0p5T"
)

gate_voltages = np.linspace(-2, 4, 241)

# Same Gate setup as your working ICE loop (terminal-safe)
configure_gate(nplc=1, current_range_A=5e-9, compliance_A=5e-9)
Gate.output_on()

for gate_volt in gate_voltages:

    # Same ramp call as the loop you sent
    Gate.voltage_ramping(
        gate_volt,
        5,
        0.1,
        # callback=lambda _v,_i: print(f"gate voltage: {_v:.4g} V, gate current: {_i*1e9:.4g} nA", end="\r")
    )

    time.sleep(3)

    hysteresis_voltage_sweep(
        420,
        -420,
        1,
        100e-6,
        GATE_LOOP_SAVE_DIR
    )

# Same ending as your working loop
Gate.voltage_ramping(0, 10, 0.1)
Gate.output_off()

print("Gate-loop IV finished.")

# Optional end-of-session cleanup


In [ ]:
# Optional safe ending
# Run only when you are finished measuring.

try:
    Gate.voltage_ramping(0, 10, 0.1)
except Exception as exc:
    print("Gate ramp-to-zero warning:", exc)

try:
    Bias.voltage_ramping(0, 10, 0.2)
except Exception as exc:
    print("Bias ramp-to-zero warning:", exc)

Gate.output_off()
Bias.output_off()

print("Gate and Bias outputs OFF.")
